# Week 5 Assignment — Apache Spark Fundamentals

**Topic:** Spark DataFrames — Data Cleaning, Transformation & Aggregation  
**Dataset:** `retail_transactions.csv` (self-generated, 530 rows)



## **Environment Setup**

In [39]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Start the Spark session
spark = SparkSession.builder.appName("Week5_Spark_Assignment").getOrCreate()

print(" SparkSession started successfully!")
print("Spark Version:", spark.version)

 SparkSession started successfully!
Spark Version: 4.0.3


* SparkSession is the entry point of every PySpark application.
* All Spark operations are performed through this session.



---
##  Load Dataset

In [40]:
FILE_PATH = "/content/retail_transactions.csv"

df = spark.read.csv(FILE_PATH, header=True, inferSchema=True)

print(" Dataset loaded!")
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")
print()
print("Column names:", df.columns)

 Dataset loaded!
Total rows: 530
Total columns: 16

Column names: ['record_id', 'user_id', 'transaction_date', 'age', 'city', 'region', 'product_category', 'sale_amount', 'price', 'store_id', 'subscription', 'status', 'email', 'username', 'raw_timestamp', 'revenue']


In [41]:
# Preview the data
df.show(10, truncate=False)

+---------+-------+----------------+---+---------+-------+----------------+-----------+-------+---------+------------+--------+-------------------+--------+-------------------+-------+
|record_id|user_id|transaction_date|age|city     |region |product_category|sale_amount|price  |store_id |subscription|status  |email              |username|raw_timestamp      |revenue|
+---------+-------+----------------+---+---------+-------+----------------+-----------+-------+---------+------------+--------+-------------------+--------+-------------------+-------+
|230      |USR1005|2023-02-06      |20 |Jaipur   |North  |Clothing        |2681.72    |1588.05|STORE_008|Basic       |Inactive|user671@example.com|user_208|2023-11-23 18:22:00|7055.25|
|461      |USR1055|2023-07-01      |52 |Mumbai   |Central|Sports          |3356.41    |984.72 |STORE_011|Premium     |Inactive|user251@example.com|user_424|2023-02-28 21:26:00|7250.63|
|285      |USR1028|2023-08-26      |58 |Chennai  |North  |Sports          |

In [42]:
# Check the schema (data types of each column)
df.printSchema()

root
 |-- record_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- revenue: double (nullable = true)



---
##       Data Cleaning: Remove Duplicates & Handle Null Values
> **Goal:** Clean the dataset by removing duplicates and dealing with missing data.


In [43]:
# Step 1: Check how many duplicates exist
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()

print(f"Total rows           : {total_rows}")
print(f"Distinct rows        : {distinct_rows}")
print(f"Duplicate rows found : {total_rows - distinct_rows}")

Total rows           : 530
Distinct rows        : 500
Duplicate rows found : 30


In [44]:
# Step 2: Remove duplicates based on user_id and transaction_date
df_deduped = df.dropDuplicates(["user_id", "transaction_date"])

print(f"Rows after removing duplicates: {df_deduped.count()}")
df_deduped.show(5)

Rows after removing duplicates: 497
+---------+-------+----------------+---+---------+------+----------------+-----------+-------+---------+------------+------+-------------------+--------+-------------------+-------+
|record_id|user_id|transaction_date|age|     city|region|product_category|sale_amount|  price| store_id|subscription|status|              email|username|      raw_timestamp|revenue|
+---------+-------+----------------+---+---------+------+----------------+-----------+-------+---------+------------+------+-------------------+--------+-------------------+-------+
|      206|USR1000|      2023-02-16| 46|    Delhi| North|           Books|      446.2|   NULL|STORE_013|     Premium|  NULL|user393@example.com|user_711|2023-01-30 05:40:00| 206.03|
|       16|USR1000|      2023-06-20| 56|Bangalore|  West|            Toys|     603.78|2079.24|STORE_005|       Basic|Active|user696@example.com|user_540|2023-03-07 01:19:00|9337.57|
|      268|USR1000|      2023-10-23| 62|   Jaipur|  Ea

In [45]:
# Step 3: Check null values in each column
from pyspark.sql.functions import col, count, when, isnan

print("Null value count per column:")
print("-" * 40)
for column in df_deduped.columns:
    null_count = df_deduped.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"  {column:25s}: {null_count} nulls")

Null value count per column:
----------------------------------------
  age                      : 18 nulls
  sale_amount              : 10 nulls
  price                    : 27 nulls
  status                   : 204 nulls
  email                    : 21 nulls
  username                 : 19 nulls
  revenue                  : 21 nulls


In [46]:
# Step 4: Fill null values with appropriate defaults
df_cleaned = df_deduped.na.fill({
    "status"     : "Unknown",
    "price"      : 0.0,
    "sale_amount": 0.0,
    "revenue"    : 0.0
})

print("Null handling done!")
print(f"Rows in cleaned DataFrame: {df_cleaned.count()}")

# Verify status nulls are gone
remaining_nulls = df_cleaned.filter(col("status").isNull()).count()
print(f"Remaining null 'status' values: {remaining_nulls}")

Null handling done!
Rows in cleaned DataFrame: 497
Remaining null 'status' values: 0


In [47]:
# Step 5: Remove rows where email is null OR username is empty
df_cleaned = df_cleaned.filter(
    col("email").isNotNull() &
    (F.trim(col("username")) != "")
)

print(f"Rows after removing bad contact records: {df_cleaned.count()}")

Rows after removing bad contact records: 458


---
##    Apply Filtering Conditions
> **Goal:** Filter rows based on age range, subscription type, category, and region.


In [48]:
# Filter 1: Age between 18 and 30 (inclusive), Premium subscription
df_premium_young = df_cleaned.filter(
    F.col("age").between(18, 30) &
    (F.col("subscription") == "Premium")
)

print(f"Young Premium subscribers (18–30): {df_premium_young.count()} rows")
df_premium_young.select("user_id", "age", "subscription", "city").show(8)

Young Premium subscribers (18–30): 45 rows
+-------+---+------------+---------+
|user_id|age|subscription|     city|
+-------+---+------------+---------+
|USR1004| 22|     Premium|   Jaipur|
|USR1011| 24|     Premium|  Kolkata|
|USR1012| 25|     Premium|   Mumbai|
|USR1012| 26|     Premium|  Kolkata|
|USR1014| 24|     Premium|  Kolkata|
|USR1014| 25|     Premium|Bangalore|
|USR1015| 26|     Premium|Ahmedabad|
|USR1017| 19|     Premium|Ahmedabad|
+-------+---+------------+---------+
only showing top 8 rows


In [49]:
# Filter 2: Only West region records
df_west = df_cleaned.filter(F.col("region") == "West")

print(f"West region rows: {df_west.count()}")
df_west.select("user_id", "region", "product_category", "sale_amount").show(8)

West region rows: 94
+-------+------+----------------+-----------+
|user_id|region|product_category|sale_amount|
+-------+------+----------------+-----------+
|USR1000|  West|            Toys|     603.78|
|USR1001|  West|            Toys|    1318.32|
|USR1002|  West|            Toys|    3647.44|
|USR1004|  West|          Sports|    3994.33|
|USR1006|  West|            Toys|    1907.97|
|USR1006|  West|       Groceries|    2430.57|
|USR1007|  West|       Groceries|    3840.85|
|USR1007|  West|       Groceries|    2779.93|
+-------+------+----------------+-----------+
only showing top 8 rows


In [50]:
# Filter 3: Electronics category with sale_amount > 1000
df_electronics_big = df_cleaned.filter(
    (F.col("product_category") == "Electronics") &
    (F.col("sale_amount") > 1000)
)

print(f"Electronics sales above 1000: {df_electronics_big.count()} rows")
df_electronics_big.select("user_id", "product_category", "sale_amount", "region").show(8)

Electronics sales above 1000: 49 rows
+-------+----------------+-----------+-------+
|user_id|product_category|sale_amount| region|
+-------+----------------+-----------+-------+
|USR1001|     Electronics|    3862.08|  North|
|USR1002|     Electronics|    3484.33|  North|
|USR1002|     Electronics|    4531.83|  North|
|USR1007|     Electronics|    3057.49|  South|
|USR1009|     Electronics|    3261.49|Central|
|USR1010|     Electronics|    2829.85|Central|
|USR1011|     Electronics|    2401.92|  North|
|USR1014|     Electronics|    1035.01|   East|
+-------+----------------+-----------+-------+
only showing top 8 rows


---
## Aggregation Functions: count, sum, avg, min, max
> **Goal:** Use aggregation functions to summarize the dataset.


In [51]:
# Overall statistics for the price column
df_price_stats = df_cleaned.agg(
    F.count("price").alias("count"),
    F.round(F.min("price"),   2).alias("min_price"),
    F.round(F.max("price"),   2).alias("max_price"),
    F.round(F.avg("price"),   2).alias("avg_price"),
    F.round(F.sum("price"),   2).alias("total_price")
)

print("Price Column — Overall Statistics:")
df_price_stats.show(truncate=False)

Price Column — Overall Statistics:
+-----+---------+---------+---------+-----------+
|count|min_price|max_price|avg_price|total_price|
+-----+---------+---------+---------+-----------+
|458  |0.0      |2999.7   |1403.76  |642921.92  |
+-----+---------+---------+---------+-----------+



In [52]:
# Aggregation on sale_amount per region
df_region_stats = df_cleaned.groupBy("region").agg(
    F.count("sale_amount").alias("num_transactions"),
    F.round(F.sum("sale_amount"),  2).alias("total_sales"),
    F.round(F.avg("sale_amount"),  2).alias("avg_sale"),
    F.round(F.min("sale_amount"),  2).alias("min_sale"),
    F.round(F.max("sale_amount"),  2).alias("max_sale")
).orderBy("total_sales", ascending=False)

print("Sales Statistics by Region:")
df_region_stats.show(truncate=False)

Sales Statistics by Region:
+-------+----------------+-----------+--------+--------+--------+
|region |num_transactions|total_sales|avg_sale|min_sale|max_sale|
+-------+----------------+-----------+--------+--------+--------+
|North  |104             |262779.09  |2526.72 |0.0     |4998.0  |
|West   |94              |220084.84  |2341.33 |0.0     |4963.83 |
|East   |88              |210931.35  |2396.95 |56.75   |4906.49 |
|South  |90              |204388.11  |2270.98 |0.0     |4966.41 |
|Central|82              |193988.39  |2365.71 |0.0     |4897.33 |
+-------+----------------+-----------+--------+--------+--------+



---
##  groupBy and Conditions on Aggregated Results
> **Goal:** Group data and apply HAVING-style conditions on the aggregated output.


In [53]:
# Count records per city — show only cities with more than 50 records
df_city_counts = (
    df_cleaned
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 50)          # this is the HAVING condition
    .orderBy("record_count", ascending=False)
)

print("Cities with more than 50 records:")
df_city_counts.show(truncate=False)

Cities with more than 50 records:
+---------+------------+
|city     |record_count|
+---------+------------+
|Bangalore|57          |
|Kolkata  |51          |
+---------+------------+



In [54]:
# Average sale amount per product category in West region
df_west_category = (
    df_cleaned
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.round(F.avg("sale_amount"), 2).alias("avg_sale_amount"))
    .orderBy("avg_sale_amount", ascending=False)
)

print("Average Sale Amount by Category (West Region Only):")
df_west_category.show(truncate=False)

Average Sale Amount by Category (West Region Only):
+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|Home Decor      |3028.03        |
|Groceries       |2656.9         |
|Clothing        |2583.01        |
|Electronics     |2299.52        |
|Toys            |2024.39        |
|Sports          |1953.73        |
|Books           |1914.2         |
|Beauty          |1818.79        |
+----------------+---------------+



---
##  Wide Transformations & Shuffle Operations
> **Goal:** Understand what shuffle means and why groupBy is a wide transformation.


In [55]:

print("Physical plan for groupBy('city').count():")
print("(Look for 'Exchange' keyword — that marks the shuffle step)")
print()
df_cleaned.groupBy("city").count().explain()

Physical plan for groupBy('city').count():
(Look for 'Exchange' keyword — that marks the shuffle step)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#6975], functions=[count(1)])
   +- Exchange hashpartitioning(city#6975, 200), ENSURE_REQUIREMENTS, [plan_id=10725]
      +- HashAggregate(keys=[city#6975], functions=[partial_count(1)])
         +- Project [city#6975]
            +- Filter (isnotnull(email#6991) AND NOT (trim(username#6993, None) = ))
               +- SortAggregate(key=[user_id#4305, transaction_date#4306], functions=[first(city#4308, false), first(email#4316, false), first(username#4317, false)])
                  +- Sort [user_id#4305 ASC NULLS FIRST, transaction_date#4306 ASC NULLS FIRST], false, 0
                     +- Exchange hashpartitioning(user_id#4305, transaction_date#4306, 200), ENSURE_REQUIREMENTS, [plan_id=10718]
                        +- SortAggregate(key=[user_id#4305, transaction_date#4306], functions=[partial_fir

In [56]:
import time

# Narrow transformation: filter (no shuffle)
start = time.time()
narrow_count = df_cleaned.filter(F.col("region") == "North").count()
narrow_time = __builtins__.round(time.time() - start, 3)

# Wide transformation: groupBy (shuffle needed)
start = time.time()
wide_count = df_cleaned.groupBy("region").count().count()
wide_time = __builtins__.round(time.time() - start, 3)

print(f"Narrow (filter) time : {narrow_time}s  → no shuffle, fast")
print(f"Wide (groupBy) time  : {wide_time}s  → shuffle across partitions")
print()
print("Wide transformations cost more because data must move across partitions/nodes.")
print("Always filter() BEFORE groupBy() to reduce the data being shuffled.")

Narrow (filter) time : 0.255s  → no shuffle, fast
Wide (groupBy) time  : 0.412s  → shuffle across partitions

Wide transformations cost more because data must move across partitions/nodes.
Always filter() BEFORE groupBy() to reduce the data being shuffled.


---
##  Schema Modification: Casting & Renaming Columns
> **Goal:** Change column data types and rename columns cleanly.


In [57]:
# Cast raw_timestamp (string) to TimestampType and rename to event_time
df_with_timestamp = (
    df_cleaned
    .withColumn("event_time", F.col("raw_timestamp").cast(TimestampType()))
    .drop("raw_timestamp")
)

print("Schema after casting raw_timestamp → event_time (TimestampType):")
df_with_timestamp.select("user_id", "event_time").printSchema()
df_with_timestamp.select("user_id", "event_time").show(5, truncate=False)

Schema after casting raw_timestamp → event_time (TimestampType):
root
 |-- user_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|USR1000|2023-01-30 05:40:00|
|USR1000|2023-03-07 01:19:00|
|USR1000|2023-02-10 04:46:00|
|USR1001|2023-11-03 22:47:00|
|USR1001|2023-06-23 22:05:00|
+-------+-------------------+
only showing top 5 rows


In [58]:
# Rename sale_amount to amount and cast age to Integer
df_schema_fixed = (
    df_with_timestamp
    .withColumnRenamed("sale_amount", "amount")
    .withColumn("age", F.col("age").cast("integer"))
)

print("Updated column names:", df_schema_fixed.columns)
print()
df_schema_fixed.select("user_id", "age", "amount").show(5)

Updated column names: ['record_id', 'user_id', 'transaction_date', 'age', 'city', 'region', 'product_category', 'amount', 'price', 'store_id', 'subscription', 'status', 'email', 'username', 'revenue', 'event_time']

+-------+---+-------+
|user_id|age| amount|
+-------+---+-------+
|USR1000| 46|  446.2|
|USR1000| 56| 603.78|
|USR1000| 62|2902.94|
|USR1001| 36|1318.32|
|USR1001| 40|3862.08|
+-------+---+-------+
only showing top 5 rows


---
##  Handle Inconsistent Data: Nulls, Empty Values, Schema Issues
> **Goal:** Identify and handle messy data issues before analysis.


In [59]:
# Full null audit — count nulls in every column
print("Complete Null Audit:")
print("=" * 45)
for column in df.columns:
    null_count = df.filter(F.col(column).isNull()).count()
    empty_count = 0
    # Check empty strings only for string-type columns
    if dict(df.dtypes)[column] == "string":
        empty_count = df.filter(F.trim(F.col(column)) == "").count()
    if null_count > 0 or empty_count > 0:
        print(f"  {column:25s} | nulls: {null_count:3d} | empty: {empty_count:3d}")
print()
print("Columns not listed above have no nulls or empty values.")

Complete Null Audit:
  age                       | nulls:  18 | empty:   0
  sale_amount               | nulls:  10 | empty:   0
  price                     | nulls:  28 | empty:   0
  status                    | nulls: 214 | empty:   0
  email                     | nulls:  22 | empty:   0
  username                  | nulls:  20 | empty:   0
  revenue                   | nulls:  23 | empty:   0

Columns not listed above have no nulls or empty values.


In [60]:
# Strategy: decide what to do with each type of bad data
# 1. Nulls in numeric columns → fill with 0
# 2. Nulls in categorical columns → fill with 'Unknown'
# 3. Rows with null email or empty username → drop them (unusable contact records)

df_handled = (
    df_deduped
    # Fill numeric nulls
    .na.fill({"price": 0.0, "sale_amount": 0.0, "revenue": 0.0, "age": 0})
    # Fill categorical nulls
    .na.fill({"status": "Unknown", "subscription": "Basic"})
    # Drop rows where contact info is broken
    .filter(
        F.col("email").isNotNull() &
        (F.trim(F.col("username")) != "")
    )
)

print(f"Rows after complete data handling: {df_handled.count()}")
print("Dataset is now clean and ready for analysis!")

Rows after complete data handling: 458
Dataset is now clean and ready for analysis!


---
##  Build a Complete Data Processing Pipeline
> **Goal:** Combine all cleaning and aggregation steps into one end-to-end pipeline.


In [61]:
# ─────────────────────────────────────────────────────────────────────────
# COMPLETE DATA PROCESSING PIPELINE
# Step 1: Load raw data
# Step 2: Remove duplicates
# Step 3: Fill null values
# Step 4: Remove invalid contact records
# Step 5: Cast and rename columns
# Step 6: Aggregate — total revenue by store
# ─────────────────────────────────────────────────────────────────────────

df_pipeline_output = (
    # Step 1 already done — df is loaded above

    # Step 2: Remove duplicates
    df.dropDuplicates()

    # Step 3: Fill nulls
    .na.fill({"price": 0.0, "revenue": 0.0, "sale_amount": 0.0, "status": "Unknown"})

    # Step 4: Remove broken contact rows
    .filter(
        F.col("email").isNotNull() &
        (F.trim(F.col("username")) != "")
    )

    # Step 5: Fix timestamp column
    .withColumn("event_time", F.col("raw_timestamp").cast(TimestampType()))
    .drop("raw_timestamp")

    # Step 6: Group by store_id and calculate total revenue
    .groupBy("store_id")
    .agg(
        F.round(F.sum("revenue"),   2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.round(F.avg("price"),     2).alias("avg_price")
    )
    .orderBy("total_revenue", ascending=False)
)

print("Final Pipeline Output — Revenue by Store:")
df_pipeline_output.show(20, truncate=False)

top = df_pipeline_output.first()
print(f"Best performing store : {top['store_id']}  | Revenue: ₹{top['total_revenue']}  | Transactions: {top['transaction_count']}")

Final Pipeline Output — Revenue by Store:
+---------+-------------+-----------------+---------+
|store_id |total_revenue|transaction_count|avg_price|
+---------+-------------+-----------------+---------+
|STORE_011|141709.81    |25               |1023.96  |
|STORE_008|129855.9     |29               |1420.77  |
|STORE_014|127601.23    |26               |1320.27  |
|STORE_015|126675.26    |24               |1418.39  |
|STORE_019|121467.67    |25               |1546.89  |
|STORE_002|117776.63    |26               |1208.97  |
|STORE_013|116060.13    |27               |1399.86  |
|STORE_006|112959.78    |22               |1209.12  |
|STORE_018|112310.67    |28               |1428.64  |
|STORE_007|108428.38    |19               |1372.78  |
|STORE_001|108349.04    |20               |1248.71  |
|STORE_010|105255.33    |24               |1316.52  |
|STORE_005|98413.6      |22               |1616.56  |
|STORE_012|97541.03     |23               |1294.73  |
|STORE_003|93647.74     |18             

In [62]:
# Pipeline summary stats
print("Pipeline Summary")
print("=" * 50)
print(f"Raw rows loaded               : {df.count()}")
print(f"After deduplication           : {df.dropDuplicates().count()}")
print(f"After cleaning (full pipeline): {df_handled.count()}")
print(f"Stores in final output        : {df_pipeline_output.count()}")

Pipeline Summary
Raw rows loaded               : 530
After deduplication           : 500
After cleaning (full pipeline): 458
Stores in final output        : 20


In [63]:
# Save the cleaned dataset
df_cleaned.coalesce(1).write.mode("overwrite").option("header", True).csv("cleaned_dataset")

# Save the final pipeline output
df_pipeline_output.coalesce(1).write.mode("overwrite").option("header", True).csv("store_revenue_summary")

print("Output files saved successfully!")

Output files saved successfully!


---
##  Stop SparkSession

In [65]:
spark.stop()
print(' SparkSession stopped. All done!')

 SparkSession stopped. All done!
